# **Redes Neuronales 2**
## Práctica 0: Conceptos básicos de PyTorch

PyTorch ofrece algunas clases para simplificar la creación, entrenamiento y uso de redes neuronales complejas. En la teoría, ya vimos los fundamentos de la programación diferenciable con ejemplos usando esta biblioteca. Ahora veremos algunos conceptos más, que nos servirán mucho para todo el curso.

---



### El [módulo](https://pytorch.org/docs/stable/generated/torch.nn.Module.html) (`torch.nn.Module`)
- Esta es la clase más importante ya que las redes neuronales que definamos deben heredar de esta clase (el submódulo `nn` se llama así por _neural network_).
- Aunque no es estrictamente requerido, en esta clase casi siempre habrá un constructor `__init__`. Este método definirá los distintos componentes de la red (con parámetros aprendibles o no) y otros atributos del objeto, los que a su vez pueden ser otras subclases de `torch.nn.Module`, permitiendo componer redes neuronales muy fácilmente. Todos estos componentes serán detectados automáticamente por PyTorch.
- Además, la clase definida debe contar con un método especial `forward`, que indica el conjunto de operaciones que sufre un dato de entrada cuando pasa por la red, hasta la salida (dada por el valor que devuelve esta función).
- El método `forward` irá construyendo el grafo computacional de manera dinámica en base a las operaciones que se vayan aplicando a nuestro dato de entrada.
- **Nota**: es esencial que todas las operaciones se realicen con funciones de `torch` para que PyTorch pueda efectivamente propagar los gradientes a través del grafo.

Veamos un ejemplo:

In [1]:
import torch
from torch import nn
device = torch.device("cpu")

In [2]:
class MiRedNeuronal(nn.Module):

    def __init__(self):
        #super(MiRedNeuronal, self).__init__()
        super().__init__()
        # Definimos una capa lineal con 3 entradas y 2 salidas
        self.fc = nn.Linear(3, 2) # fc por "fully connected"

    def forward(self, x):
        # Aplicamos la transformación lineal (sin función de activación)
        y = self.fc(x)
        return y

red = MiRedNeuronal()
print(red)

MiRedNeuronal(
  (fc): Linear(in_features=3, out_features=2, bias=True)
)


In [3]:
print(red.fc)
print(red.fc.bias)
print(red.fc.in_features)
print(red.fc.out_features)

print(red.fc.weight)


Linear(in_features=3, out_features=2, bias=True)
Parameter containing:
tensor([-0.5225, -0.5214], requires_grad=True)
3
2
Parameter containing:
tensor([[ 0.1598, -0.3067,  0.4639],
        [ 0.2328,  0.4618,  0.5228]], requires_grad=True)


In [4]:
print(f"El número de parámetros del modelo es: {sum([torch.numel(x) for x in red.parameters()])}")

El número de parámetros del modelo es: 8


Examinemos ahora los valores que trae la red al ser creada:

In [5]:
# Mostrando los pesos iniciales de la red
print(f"Pesos iniciales: {red.fc.weight=}")
print(f"Bias inicial: {red.fc.bias=}\n")

Pesos iniciales: red.fc.weight=Parameter containing:
tensor([[ 0.1598, -0.3067,  0.4639],
        [ 0.2328,  0.4618,  0.5228]], requires_grad=True)
Bias inicial: red.fc.bias=Parameter containing:
tensor([-0.5225, -0.5214], requires_grad=True)



Si no queremos que se muestre el `requires_grad=True` en la salida podemos hacer `.detach()`. De esta forma estamos desvinculando al tensor del grafo computacional, convirtiéndolo en un tensor independiente que ya no participará en la retropropagación.

Probemos aplicar la red a un tensor cualquiera (de la dimensión correcta):

In [6]:
# Probando con un tensor de entrada (con dtype explícito)
x = torch.tensor([[1.0, 2.0, 3.0]], dtype=torch.float32)  # Un solo ejemplo con tres características. Notar el doble corchete! ¿Por qué lo usamos?
y = red.forward(x) # acá en realidad estamos llamando al método forward
print(f"Salida de la red: {y=}")

Salida de la red: y=tensor([[0.4155, 2.2036]], grad_fn=<AddmmBackward0>)


#### **Algunas utilidades adicionales de `torch.nn`**

Funciones de activación predefinidas:

In [7]:
import torch
from torch import nn

x = torch.tensor([3, -4, 5, 0, 1e-4])
relu = nn.ReLU()
tanh = nn.Tanh()
sigmoid = nn.Sigmoid()
leaky_relu = nn.LeakyReLU(negative_slope=0.01) # Notar qué le pasa al valor negativo abajo

print(f"{x = },\n{relu(x) = },\n{tanh(x) = },\n{sigmoid(x) = }\n{2 * sigmoid(2*x) - 1 = }\n{leaky_relu(x) = }\n")

x = tensor([ 3.0000e+00, -4.0000e+00,  5.0000e+00,  0.0000e+00,  1.0000e-04]),
relu(x) = tensor([3.0000e+00, 0.0000e+00, 5.0000e+00, 0.0000e+00, 1.0000e-04]),
tanh(x) = tensor([ 9.9505e-01, -9.9933e-01,  9.9991e-01,  0.0000e+00,  1.0000e-04]),
sigmoid(x) = tensor([0.9526, 0.0180, 0.9933, 0.5000, 0.5000])
2 * sigmoid(2*x) - 1 = tensor([ 9.9505e-01, -9.9933e-01,  9.9991e-01,  0.0000e+00,  1.0002e-04])
leaky_relu(x) = tensor([ 3.0000e+00, -4.0000e-02,  5.0000e+00,  0.0000e+00,  1.0000e-04])



La forma anterior requiere **instanciar** las funciones de activación. `nn.ReLU` no es una ReLU sino una "fábrica" de ReLUs.

Otra forma de hacerlo que no requiere instanciación es la siguiente:

In [8]:
import torch.nn.functional as F

print(f"{x = },\n{F.relu(x) = },\n{F.tanh(x) = },\n{F.sigmoid(x) = }\n{F.leaky_relu(x, 0.01) = }")

x = tensor([ 3.0000e+00, -4.0000e+00,  5.0000e+00,  0.0000e+00,  1.0000e-04]),
F.relu(x) = tensor([3.0000e+00, 0.0000e+00, 5.0000e+00, 0.0000e+00, 1.0000e-04]),
F.tanh(x) = tensor([ 9.9505e-01, -9.9933e-01,  9.9991e-01,  0.0000e+00,  1.0000e-04]),
F.sigmoid(x) = tensor([0.9526, 0.0180, 0.9933, 0.5000, 0.5000])
F.leaky_relu(x, 0.01) = tensor([ 3.0000e+00, -4.0000e-02,  5.0000e+00,  0.0000e+00,  1.0000e-04])


Finalmente, podemos usar **funciones de pérdida** predefinidas:

In [9]:
mse = nn.MSELoss(reduction="mean") # puede ser "sum" también

x = torch.tensor([[1.0, 2.0, 3.0], [1.0, 3.0, 4.0]], dtype=torch.float32, requires_grad=True)  # Un solo ejemplo con tres características
y = torch.tensor([[2.0, 4.0], [1.0, 5.0]], dtype=torch.float32) # esto sería la salida esperada (la y_d de la teoría)
y_pred = red.forward(x)

loss = mse(y, y_pred)
print(loss) # <- notar que guarda los gradientes!

loss.backward()

tensor(2.3008, grad_fn=<MseLossBackward0>)


---

In [10]:
print(red.fc.weight.grad)
red.fc.bias.grad

tensor([[-1.0059, -2.2255, -3.2315],
        [-1.8041, -4.5141, -6.3183]])


tensor([-1.0059, -1.8041])

 ### El [Dataset](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset) (`torch.utils.data.Dataset`)

La segunda clase importante de PyTorch tiene que ver con _los datos_, y es el `Dataset` (`torch.utils.data.Dataset`). Nuevamente, esta es una clase abstracta, y para manejar nuestros datos necesitaremos definir una clase propia que herede de esta.

**Nota de nomenclatura:** nos referiremos a "dataset" en el sentido más general y a "`Dataset`" para indicar la clase de PyTorch en particular.

En este caso, se requiere definir dos métodos especiales en nuestra subclase, `__getitem__` y `__len__`.
- El primero recibe un "índice" y devuelve el dato o ejemplo de entrenamiento asociado a ese índice. El índice normalmente es un entero, pero no tiene por qué ser así ya que PyTorch no lo impone. Uno puede pasar el objeto que quiera como índice siempre que implemente la lógica correcta en `__getitem__` (y en otra clase que veremos luego, el `DataLoader`).
- Lo mismo ocurre con el valor que devuelve `__getitem__`: _puede_ ser "cualquier cosa", pero normalmente será un tensor, una tupla de tensores como `(x, y)` (entradas y etiquetas) o un diccionario `{"x": x, "y": y}`. Esto dependerá de cómo implementemos el resto de la lógica en nuestro entrenamiento.
- El segundo método, `__len__`, devuelve la longitud del dataset, esto es, el número total de datos o ejemplos que tiene.
- Como antes, el constructor `__init__` no es requerido pero casi siempre estará presente para definir parámetros del dataset, como por ejemplo las rutas a los diferentes archivos de datos.

![Tomado del libro _Deep Learning with PyTorch_](https://drek4537l1klr.cloudfront.net/stevens2/Figures/CH07_F02_Stevens2_GS.png)

**Nota:** cómo hacemos exactamente el manejo de los datos dependerá del caso particular. A veces podremos cargar todo en memoria RAM y simplemente acceder a las "rebanadas" necesarias en cada paso del entrenamiento. Por otro lado, si el dataset es muy grande y no entra en memoria, puede que tengamos que cargarlas de disco en cada iteración, lo cual introducirá una latencia adicional.

**Ejemplo**: nuestro dataset está en un archivo `csv`, donde cada fila representa un dato (siendo $m_1$ columnas la entrada o _características_, y $m_2$ columnas la salida o _etiquetas_ a predecir). Podemos implementar `__getitem__` de manera que el índice identifique la fila a extraer, y devuelva una tupla `(x,y)`, con `x` e `y` tensores unidimensionales de longitud $m_1$ y $m_2$, respectivamente. `__len__` devolverá entonces el número de filas del archivo (excluyendo el encabezado, claro).

In [11]:
import os
import torch
from torch.utils.data import Dataset
import pandas as pd
from typing import Tuple


In [12]:
class MiDataset(Dataset):

    FEATURE_COLUMNS = ["x1", "x2", "x3"]
    LABEL_COLUMNS  = ["y1", "y2"]

    def __init__(self, file: str):
        self.data = pd.read_csv(file)

    def __getitem__(self, idx: int) -> Tuple:

        row = self.data.iloc[idx] # accedemos a la idx'ésima fila del dataframe
        x = torch.tensor(row[self.FEATURE_COLUMNS].values, dtype=torch.float32) # ¿Por qué necesitamos especificar el dtype?
        y = torch.tensor(row[self.LABEL_COLUMNS].values, dtype=torch.float32)
        return x, y

        # otra opción:
        # return dict(x=x, y=y) # habría que cambiar el "type hint" de Tuple por Dict arriba


    def __len__(self):
        return self.data.shape[0]

        # también podría ser simplemente
        # return len(self.data)


Probemos este `Dataset` con nuestra "red neuronal" de antes:

In [13]:
import pandas as pd
import numpy as np

In [14]:
import sys
print(sys.executable)

d:\facuult\REDES_NEURONALES_II\.venv\Scripts\python.exe


In [15]:
import pandas as pd

print(pd)
print(pd.__file__)
print(hasattr(pd, "DataFrame"))

<module 'pandas' from 'd:\\facuult\\REDES_NEURONALES_II\\.venv\\Lib\\site-packages\\pandas\\__init__.py'>
d:\facuult\REDES_NEURONALES_II\.venv\Lib\site-packages\pandas\__init__.py
True


In [16]:
# Creamos un dataset random y lo guardamos en un csv, pues nuestro MiDataset necesita un csv
num_samples = 1000

data = { columna: np.random.randn(num_samples) for columna in ["x1", "x2", "x3", "x4", "y1", "y2", "y3"] }

# Crear DataFrame y guardarlo en CSV
df = pd.DataFrame(data)
df.to_csv("dataset.csv", index=False)

In [21]:
print(df)

           x1        x2        x3        x4        y1        y2        y3
0    0.048600 -0.845449 -0.500831  1.106034 -0.640291  1.059504  0.985312
1    0.374732  0.186785 -0.798902  1.703297 -0.466580  0.968018  0.023355
2   -1.675046  0.267845 -0.013372 -0.969904  0.650916 -1.385617  0.486183
3    0.262101  1.390221  0.199540  0.640576 -1.115646 -0.900537  1.314008
4   -1.398747 -0.513580  1.369536  1.368737 -0.453922 -0.151888  0.269393
..        ...       ...       ...       ...       ...       ...       ...
995  1.827804  0.155692 -0.491035  1.340180 -0.841447  1.126973  0.837065
996  1.107350  1.247102  0.090100  0.881084 -1.305982  0.857025  0.003213
997  1.106494 -2.353366 -0.409145  1.490018 -1.716071 -0.794094  1.256337
998  0.552103  0.311686  1.251289 -1.619681  0.349088  1.246661  1.874241
999 -0.356663 -0.120757 -1.377813  3.116533 -0.035444 -1.390234 -1.032834

[1000 rows x 7 columns]


In [23]:
# Ahora cargamos nuestro dataset
dataset = MiDataset("dataset.csv") #llamamos el construcot

# extraemos un dato (el primero)
x, y = dataset[0]

# lo mostramos
print(f"{x = }\n{y = }")

# y lo pasamos por la red
yp = red.forward(x)
print(f"Predicción de MiRedNeuronal: {yp.detach()}")

x = tensor([ 0.0486, -0.8454, -0.5008])
y = tensor([-0.6403,  1.0595])
Predicción de MiRedNeuronal: tensor([-0.4878, -1.1624])


---

### El [Dataloader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) (`torch.utils.data.DataLoader`)

Ahora, necesitamos una clase adicional, el `DataLoader` (`torch.utils.data.DataLoader`). Esta clase actúa como un iterador sobre el `Dataset`, aplicando lógica adicional como el reordenamiento de los datos, la paralelización del acceso y la generación de lotes (_batches_ o _minibatches_) sobre los cuales iremos calculando el gradiente.
Un parámetro importante a la hora de inicializar este objeto es el `batch_size` o **tamaño del lote**, es decir, el número de ejemplos que contendrá.

Hay dos elementos principales: el `Sampler` y el `DataCollator`.
- El `Sampler` determina el orden en que se visitan los datos del `Dataset`.
- El `DataCollator` indica cómo se fusionan los distintos elementos extraídos del `Dataset` en un único _batch_ o lote.

A menudo, las opciones por omisión para estos dos elementos ya resuelven el problema de la manera deseada. Tomemos como ejemplo `MiDataset`, que provee una tupla de $\mathbf{x}\in\mathbb{R}^3$ e $\mathbf{y}\in\mathbb{R}^2$. Ahora bien:
- ¿Qué nos gustaría que haga el `Dataloader` para armar los lotes a partir de estos objetos? Probablemente que los apile a lo largo de una nueva dimensión y nos devuelva una tupla de dos tensores, $\mathbf{X}\in\mathbb{R}^{N\times3}$ e $\mathbf{Y}\in\mathbb{R}^{N\times2}$, donde $N$ es el `batch_size`. Este es exactamente el comportamiento por omisión del `DataCollator`.
- ¿En qué orden podría visitar los datos el `DataLoader`? La opción más trivial sería permutar al azar los índices (que van desde `0` hasta `len(dataset)-1`) e ir tomando rebanadas de longitud `batch_size` hasta agotar el dataset. Este es exactamente el comportamiento del `Sampler` cuando ponemos `shuffle=True`; en caso contrario se visitan los elementos en el orden original.

Nota: en este ejemplo se hace evidente que necesitamos poder calcular la longitud del `Dataset`, lo que justifica que tengamos que definir `__len__` en esa clase, como vimos más arriba.

Sin embargo, existen escenarios más complicados en los que puede ser necesario definirlos de manera personalizada, por ejemplo:
- **Datos de longitud variable**: cuando los ejemplos tienen distinta longitud y no se pueden concatenar directamente, en el caso del `DataCollator`,
- **Índices no enteros**: cuando definimos el índice como un objeto sobre el cual no es claro cómo muestrear valores, en el caso del `Sampler`. En este caso hay que definir este comportamiento.

In [24]:
# Usemos el dataloader en un ejemplo sencillo
from torch.utils.data import DataLoader

# dataset viene de la celda anterior
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

In [ ]:
x, y = next(iter(dataloader))
print(x.shape, y.shape) #tiene 8 batches

torch.Size([8, 3]) torch.Size([8, 2])


---

### Entrenando por batches

Ahora usemos las clases anteriores para entrenar nuestra red neuronal. En este caso no va a haber aprendizaje pues en el dataset sintético que hicimos no hay dependencias entre $x$ e $y$, pero les va a resultar útil como referencia para los ejercicios de las próximas prácticas.

In [30]:
from torch.optim import SGD
from tqdm import tqdm # para la barra de progreso

In [31]:
mse = nn.MSELoss(reduction="mean")
optimizador = SGD(red.parameters(), lr=0.001)

for epoca in tqdm(range(100)): #100 epocas
    # vimos que el dataloader es un iterable, que en cada iteración devuelve un batch
    # entonces podemos hacer un bucle for sobre el mismo para recorrer el dataset
    for batch in dataloader:
        x, y = batch #extraemos el batch
        y_pred = red(x)
        loss = mse(y, y_pred)
        loss.backward()
        optimizador.step()
        optimizador.zero_grad()

100%|██████████| 100/100 [00:57<00:00,  1.75it/s]


Es importante notar que en cada `batch` hay `batch_size` pares entrada-salida, `(x,y)`. En este caso, el `forward` procesará todos estos ejemplos a la vez, la `loss` los agrega (recordar el argumento `reduction` de la `MSELoss`) y los gradientes se computan sobre la loss acumulada sobre todos estos ejemplos. Finalmente se actualizarán los pesos una vez por cada lote con el **gradiente promedio** en el mismo (si usamos `reduction=mean`), en lugar de hacerse una vez por ejemplo de entrenamiento (como en el **gradiente instantáneo** que vimos en la teoría).

Por otro lado, notar un detalle importante de implementación: hay que reiniciar a cero los gradientes con `.zero_grad()` en cada iteración una vez realizada la actualización de pesos $\textbf{w}$ con `.step()`, ya que de otro modo se van acumulando continuamente con cada llamada a `.backward()` en lugar de "pisar" los gradientes del paso anterior, que podría ser el comportamiento esperado.

---

### Usando GPU en PyTorch

En Colab, pueden usar computadoras con GPU de manera gratuita (con limitaciones de tiempo). Para hacerlo, hacer clic en "Opciones de conexión adicionales" -> "Cambiar tipo de entorno de ejecución", y elegir un tipo de entorno con GPU. Por ejemplo, **T4 GPU**.

PyTorch permite realizar cálculos en GPU muy fácilmente. Simplemente, hay que pasar el modelo y los datos a la memoria de GPU con `.to(device)`, donde `device` es un `string` que identifica a la GPU, y hacer lo mismo con los tensores de datos.

In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
import torch
from torch import nn

class MiRedNeuronal(nn.Module):

    def __init__(self):
        super(MiRedNeuronal, self).__init__()
        # Definimos una capa lineal con 3000 entradas y 2000 salidas
        self.fc = nn.Linear(3000, 2000) # fc por "fully connected"

    def forward(self, x):
        # Aplicamos la transformación lineal (sin función de activación)
        y = self.fc(x)
        return y

red = MiRedNeuronal()
print(red)

MiRedNeuronal(
  (fc): Linear(in_features=3000, out_features=2000, bias=True)
)


In [ ]:
# Primero probemos en la CPU
# (tanto modelos como datos se crean inicialmente en CPU)

# Generamos datos al azar
x = torch.randn(3000).unsqueeze(0)

In [ ]:
# Usemos %%timeit para comparar los tiempos de ejecución (En Colab. Saltear en local)
%%timeit
red(x)

1.11 ms ± 50.3 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [ ]:
# Y ahora en GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [ ]:
# Pasamos los datos a la GPU
x = x.to(device)

In [ ]:
# Pasamos el modelo a la GPU
red = red.to(device)

In [ ]:
# Usemos %%timeit para comparar los tiempos de ejecución
%%timeit
red(x)

98.7 µs ± 102 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)
